# Data Analysis Bootcamp — Lesson 14
## Advanced Data Reshaping, Merge, Join & Concatenation

Complete hands-on Jupyter Notebook.

### Learning Objectives
By the end of this lesson, you will understand:
1. Why multiple DataFrames exist in real projects
2. `pd.concat()`
3. `merge()`
4. `join()`
5. Inner, left, right, and outer joins
6. One-to-one, one-to-many, and many-to-many relationships
7. Joining on different column names
8. Multiple-key joins
9. `indicator=True`
10. `validate=`
11. Handling duplicate keys
12. Avoiding accidental row multiplication
13. Practical manufacturing/QA data integration

## 1. Why Do We Need Multiple DataFrames?

Real-world data rarely lives in one table.

A manufacturing organization may have separate datasets for:

- Production
- Inspection
- Defects
- Part master
- Customers
- Suppliers
- Shifts

A Data Analyst often needs to integrate these sources into one analysis-ready dataset.

In [1]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt

tips = sns.load_dataset("tips")
tips.head()

,total_bill,tip,sex,smoker,day,time,size
0,16.99,1.01,Female,No,Sun,Dinner,2
1,10.34,1.66,Male,No,Sun,Dinner,3
2,21.01,3.50,Male,No,Sun,Dinner,3
3,23.68,3.31,Male,No,Sun,Dinner,2
4,24.59,3.61,Female,No,Sun,Dinner,4


## 2. Create Customer and Order Data

In [2]:
customers = pd.DataFrame({
    "Customer_ID": [101, 102, 103, 104],
    "Customer_Name": ["Alice", "Bob", "Charlie", "David"],
    "City": ["Delhi", "Mumbai", "Pune", "Lucknow"]
})

customers

,Customer_ID,Customer_Name,City
0,101,Alice,Delhi
1,102,Bob,Mumbai
2,103,Charlie,Pune
3,104,David,Lucknow


In [3]:
orders = pd.DataFrame({
    "Order_ID": [1, 2, 3, 4, 5],
    "Customer_ID": [101, 102, 101, 104, 105],
    "Amount": [500, 700, 300, 900, 450]
})

orders

,Order_ID,Customer_ID,Amount
0,1,101,500
1,2,102,700
2,3,101,300
3,4,104,900
4,5,105,450


Notice:

- `Customer_ID = 105` exists in `orders` but not in `customers`.
- `Customer_ID = 103` exists in `customers` but has no order.

These mismatches will help us understand joins.

## 3. What Is a Key?

A key is a column used to identify or connect records between datasets.

Here:

```text
customers.Customer_ID
orders.Customer_ID
```

are the common keys.

## 4. `pd.concat()` — Vertical Concatenation

In [4]:
sales_jan = pd.DataFrame({
    "Date": ["2026-01-01", "2026-01-02"],
    "Sales": [100, 120]
})

sales_feb = pd.DataFrame({
    "Date": ["2026-02-01", "2026-02-02"],
    "Sales": [130, 150]
})

sales_jan, sales_feb

(         Date  Sales
 0  2026-01-01    100
 1  2026-01-02    120,
          Date  Sales
 0  2026-02-01    130
 1  2026-02-02    150)

In [5]:
pd.concat([sales_jan, sales_feb])

,Date,Sales
0,2026-01-01,100
1,2026-01-02,120
0,2026-02-01,130
1,2026-02-02,150


This stacks rows. Use `ignore_index=True` when you want a fresh sequential index.

In [6]:
pd.concat(
    [sales_jan, sales_feb],
    ignore_index=True
)

,Date,Sales
0,2026-01-01,100
1,2026-01-02,120
2,2026-02-01,130
3,2026-02-02,150


## 5. Horizontal Concatenation

In [7]:
left = pd.DataFrame({"A": [1, 2, 3]})
right = pd.DataFrame({"B": [10, 20, 30]})

pd.concat([left, right], axis=1)

,A,B
0,1,10
1,2,20
2,3,30


Remember:

- `axis=0` → combine rows
- `axis=1` → combine columns

Simple mental model:

**concat = put DataFrames together**

**merge = match records using a key**

## 6. `merge()` — Combining DataFrames by Key

In [8]:
merged = pd.merge(
    orders,
    customers,
    on="Customer_ID")

merged

,Order_ID,Customer_ID,Amount,Customer_Name,City
0,1,101,500,Alice,Delhi
1,2,102,700,Bob,Mumbai
2,3,101,300,Alice,Delhi
3,4,104,900,David,Lucknow


By default, `pd.merge()` uses an **inner join** when `how` is not specified.

## 7. Inner Join

In [9]:
pd.merge(
    orders,
    customers,
    on="Customer_ID",
    how="inner")

,Order_ID,Customer_ID,Amount,Customer_Name,City
0,1,101,500,Alice,Delhi
1,2,102,700,Bob,Mumbai
2,3,101,300,Alice,Delhi
3,4,104,900,David,Lucknow


An inner join keeps only keys present in both DataFrames.

Common keys here are:

```text
101
102
104
```

Therefore:
- Customer 103 is excluded because there is no order.
- Order 5 / Customer 105 is excluded because there is no customer record.

## 8. Left Join

In [10]:
pd.merge(
    orders,
    customers,
    on="Customer_ID",
    how="left")

,Order_ID,Customer_ID,Amount,Customer_Name,City
0,1,101,500,Alice,Delhi
1,2,102,700,Bob,Mumbai
2,3,101,300,Alice,Delhi
3,4,104,900,David,Lucknow
4,5,105,450,NaN,NaN


A left join keeps **everything from the left DataFrame**.

Since `orders` is on the left, all orders remain. Customer 105 gets missing customer information.

## 9. Right Join

In [11]:
pd.merge(
    orders,
    customers,
    on="Customer_ID",
    how="right")

,Order_ID,Customer_ID,Amount,Customer_Name,City
0,1.0,101,500.0,Alice,Delhi
1,3.0,101,300.0,Alice,Delhi
2,2.0,102,700.0,Bob,Mumbai
3,NaN,103,NaN,Charlie,Pune
4,4.0,104,900.0,David,Lucknow


A right join keeps everything from the right DataFrame. Customer 103 remains even though they have no order.

## 10. Outer Join

In [12]:
pd.merge(
    orders,
    customers,
    on="Customer_ID",
    how="outer")

,Order_ID,Customer_ID,Amount,Customer_Name,City
0,1.0,101,500.0,Alice,Delhi
1,3.0,101,300.0,Alice,Delhi
2,2.0,102,700.0,Bob,Mumbai
3,NaN,103,NaN,Charlie,Pune
4,4.0,104,900.0,David,Lucknow
5,5.0,105,450.0,NaN,NaN


An outer join keeps everything from both DataFrames.

Mental model:

- `inner` → intersection
- `left` → all left + matches from right
- `right` → all right + matches from left
- `outer` → union of both

## 11. `indicator=True` — Join Audit

In [13]:
merge_check = pd.merge(
    orders,
    customers,
    on="Customer_ID",
    how="outer",
    indicator=True)

merge_check

,Order_ID,Customer_ID,Amount,Customer_Name,City,_merge
0,1.0,101,500.0,Alice,Delhi,both
1,3.0,101,300.0,Alice,Delhi,both
2,2.0,102,700.0,Bob,Mumbai,both
3,NaN,103,NaN,Charlie,Pune,right_only
4,4.0,104,900.0,David,Lucknow,both
5,5.0,105,450.0,NaN,NaN,left_only


The `_merge` column contains:

- `both`
- `left_only`
- `right_only`

This is very useful for data-quality audits.

In [14]:
merge_check[merge_check["_merge"] == "left_only"]

,Order_ID,Customer_ID,Amount,Customer_Name,City,_merge
5,5.0,105,450.0,NaN,NaN,left_only


In [15]:
merge_check[merge_check["_merge"] == "right_only"]

,Order_ID,Customer_ID,Amount,Customer_Name,City,_merge
3,NaN,103,NaN,Charlie,Pune,right_only


## 12. Manufacturing Example — Production + Inspection

In [16]:
production = pd.DataFrame({
    "Part_ID": ["P001", "P002", "P003", "P004", "P005"],
    "Line": ["A", "A", "B", "B", "C"],
    "Quantity": [500, 600, 450, 700, 300]
})

production

,Part_ID,Line,Quantity
0,P001,A,500
1,P002,A,600
2,P003,B,450
3,P004,B,700
4,P005,C,300


In [17]:
inspection = pd.DataFrame({
    "Part_ID": ["P001", "P002", "P003", "P004"],
    "Measurement": [10.01, 10.08, 9.99, 10.03],
    "Result": ["Pass", "Fail", "Pass", "Pass"]
})

inspection

,Part_ID,Measurement,Result
0,P001,10.01,Pass
1,P002,10.08,Fail
2,P003,9.99,Pass
3,P004,10.03,Pass


In [18]:
production_inspection = pd.merge(
    production,
    inspection,
    on="Part_ID",
    how="left")

production_inspection

,Part_ID,Line,Quantity,Measurement,Result
0,P001,A,500,10.01,Pass
1,P002,A,600,10.08,Fail
2,P003,B,450,9.99,Pass
3,P004,B,700,10.03,Pass
4,P005,C,300,NaN,NaN


P005 remains in the production data but has no inspection record.

This can be a meaningful QA data-quality finding.

In [19]:
production_inspection[
    production_inspection["Result"].isnull()]

,Part_ID,Line,Quantity,Measurement,Result
4,P005,C,300,NaN,NaN


## 13. Add Part Master Data

In [20]:
part_master = pd.DataFrame({
    "Part_ID": ["P001", "P002", "P003", "P004", "P005"],
    "Product": ["Sensor", "Sensor", "Module", "Module", "Board"],
    "Customer": ["Customer_A", "Customer_B", "Customer_A",
                 "Customer_C", "Customer_B"]
})

part_master

,Part_ID,Product,Customer
0,P001,Sensor,Customer_A
1,P002,Sensor,Customer_B
2,P003,Module,Customer_A
3,P004,Module,Customer_C
4,P005,Board,Customer_B


In [21]:
final_data = production_inspection.merge(
    part_master,
    on="Part_ID",
    how="left")

final_data

,Part_ID,Line,Quantity,Measurement,Result,Product,Customer
0,P001,A,500,10.01,Pass,Sensor,Customer_A
1,P002,A,600,10.08,Fail,Sensor,Customer_B
2,P003,B,450,9.99,Pass,Module,Customer_A
3,P004,B,700,10.03,Pass,Module,Customer_C
4,P005,C,300,NaN,NaN,Board,Customer_B


We have now integrated:

```text
Production
+
Inspection
+
Part Master
```

This is a typical data-integration workflow.

## 14. Merge Using Different Column Names

In [22]:
customers2 = customers.rename(
    columns={"Customer_ID": "Cust_ID"}
)

customers2

,Cust_ID,Customer_Name,City
0,101,Alice,Delhi
1,102,Bob,Mumbai
2,103,Charlie,Pune
3,104,David,Lucknow


In [23]:
pd.merge(
    orders,
    customers2,
    left_on="Customer_ID",
    right_on="Cust_ID",
    how="left"
)

,Order_ID,Customer_ID,Amount,Cust_ID,Customer_Name,City
0,1,101,500,101.0,Alice,Delhi
1,2,102,700,102.0,Bob,Mumbai
2,3,101,300,101.0,Alice,Delhi
3,4,104,900,104.0,David,Lucknow
4,5,105,450,NaN,NaN,NaN


Use `left_on` and `right_on` when the key columns have different names.

## 15. Multiple-Key Merge

In [24]:
df1 = pd.DataFrame({
    "Line": ["A", "A", "B"],
    "Shift": [1, 2, 1],
    "Quantity": [500, 600, 450]
})

df2 = pd.DataFrame({
    "Line": ["A", "A", "B"],
    "Shift": [1, 2, 1],
    "Defects": [10, 15, 5]
})

df1, df2

(  Line  Shift  Quantity
 0    A      1       500
 1    A      2       600
 2    B      1       450,
   Line  Shift  Defects
 0    A      1       10
 1    A      2       15
 2    B      1        5)

In [25]:
pd.merge(
    df1,
    df2,
    on=["Line", "Shift"],
    how="left"
)

,Line,Shift,Quantity,Defects
0,A,1,500,10
1,A,2,600,15
2,B,1,450,5


Sometimes one column is not enough to identify a record.

Here, the business key is:

```text
Line + Shift
```

Always identify the correct business key before merging.

## 16. One-to-One, One-to-Many, Many-to-Many

### One-to-One

Each key occurs once in each table.

Example:

```text
Employee_ID → Employee record
```

### One-to-Many

One record on one side can match many records on the other.

Example:

```text
Customer → Orders
```

One customer can have many orders.

### Many-to-Many

Many records on both sides can match each other.

Example:

```text
Student ↔ Course
```

Many-to-many merges can produce many rows and require extra care.

## 17. The Dangerous Problem — Row Multiplication

In [26]:
left_dup = pd.DataFrame({
    "ID": [1, 1],
    "Value_A": [10, 20]
})

right_dup = pd.DataFrame({
    "ID": [1, 1],
    "Value_B": [100, 200]
})

left_dup, right_dup

(   ID  Value_A
 0   1       10
 1   1       20,
    ID  Value_B
 0   1      100
 1   1      200)

In [27]:
pd.merge(
    left_dup,
    right_dup,
    on="ID"
)

,ID,Value_A,Value_B
0,1,10,100
1,1,10,200
2,1,20,100
3,1,20,200


Both sides contain duplicate key `1`.

Each left row can match each right row, so 2 × 2 = 4 rows.

This is dangerous because downstream sums, counts, and KPIs can become incorrect.

## 18. `validate=` — Protect Your Merge

In [28]:
# One-to-one validation
pd.merge(
    df1,
    df2,
    on=["Line", "Shift"],
    validate="one_to_one"
)

,Line,Shift,Quantity,Defects
0,A,1,500,10
1,A,2,600,15
2,B,1,450,5


If the actual relationship violates the validation rule, pandas raises an error.

Available relationship checks include:

- `"one_to_one"`
- `"one_to_many"`
- `"many_to_one"`
- `"many_to_many"`

## 19. Customer → Orders: Many-to-One from Orders' Perspective

In [29]:
pd.merge(
    orders,
    customers,
    on="Customer_ID",
    how="left",
    validate="many_to_one"
)

,Order_ID,Customer_ID,Amount,Customer_Name,City
0,1,101,500,Alice,Delhi
1,2,102,700,Bob,Mumbai
2,3,101,300,Alice,Delhi
3,4,104,900,David,Lucknow
4,5,105,450,NaN,NaN


The orders table can contain many rows for one customer, while the customer table should contain one row per customer.

Therefore, from `orders` to `customers`, the relationship is:

**many-to-one**.

## 20. `join()`

In [30]:
left_indexed = pd.DataFrame({
    "Name": ["A", "B", "C"]
}, index=[101, 102, 103])

right_indexed = pd.DataFrame({
    "Score": [90, 85, 95]
}, index=[101, 102, 103])

left_indexed.join(right_indexed)

,Name,Score
101,A,90
102,B,85
103,C,95


`join()` is especially convenient when the matching key is the index.

For beginners, focus primarily on `merge()` because it is more flexible and commonly used for column-based data integration.

## 21. `merge()` vs `join()`

### `merge()`
Best when matching using explicit columns:

```python
pd.merge(df1, df2, on="ID")
```

### `join()`
Convenient when matching through indexes:

```python
df1.join(df2)
```

## 22. Data Quality Checks Before a Merge

In [31]:
# Check key data types
production["Part_ID"].dtype

dtype('O')

In [32]:
inspection["Part_ID"].dtype

dtype('O')

In [33]:
# Missing keys
production["Part_ID"].isnull().sum()

np.int64(0)

In [34]:
# Duplicate keys
production["Part_ID"].duplicated().sum()

np.int64(0)

In [35]:
# Number of unique keys
production["Part_ID"].nunique()

5

Before merging, ask:

1. Are key data types compatible?
2. Are keys missing?
3. Are keys duplicated?
4. Should the key be unique?
5. What is the expected business relationship?

## 23. Row Count Validation

In [36]:
len(production)

5

In [37]:
merged = production.merge(
    inspection,
    on="Part_ID",
    how="left"
)

len(merged)

5

If you expected one inspection record per part, an unexpected increase in row count is a strong signal that duplicate keys may exist.

## 24. Business Total Validation

In [38]:
production["Quantity"].sum()

np.int64(2550)

In [39]:
merged["Quantity"].sum()

np.int64(2550)

If a merge unexpectedly changes a business total, investigate duplicate keys and row multiplication before continuing the analysis.

## 25. Integrated QA Dataset with Validation

In [40]:
production = pd.DataFrame({
    "Part_ID": ["P001", "P002", "P003", "P004", "P005"],
    "Line": ["A", "A", "B", "B", "C"],
    "Shift": [1, 1, 2, 2, 1],
    "Produced": [500, 600, 450, 700, 300]
})

inspection = pd.DataFrame({
    "Part_ID": ["P001", "P002", "P003", "P004"],
    "Measurement": [10.01, 10.08, 9.99, 10.03],
    "Defect": [0, 1, 0, 0]
})

part_master = pd.DataFrame({
    "Part_ID": ["P001", "P002", "P003", "P004", "P005"],
    "Product": ["Sensor", "Sensor", "Module", "Module", "Board"],
    "Customer": ["A", "B", "A", "C", "B"]
})

In [41]:
qa_data = (
    production
    .merge(
        inspection,
        on="Part_ID",
        how="left",
        validate="one_to_one"
    )
    .merge(
        part_master,
        on="Part_ID",
        how="left",
        validate="one_to_one"
    )
)

qa_data

,Part_ID,Line,Shift,Produced,Measurement,Defect,Product,Customer
0,P001,A,1,500,10.01,0.0,Sensor,A
1,P002,A,1,600,10.08,1.0,Sensor,B
2,P003,B,2,450,9.99,0.0,Module,A
3,P004,B,2,700,10.03,0.0,Module,C
4,P005,C,1,300,NaN,NaN,Board,B


## 26. Find Parts Without Inspection

In [42]:
qa_data[
    qa_data["Measurement"].isnull()
]

,Part_ID,Line,Shift,Produced,Measurement,Defect,Product,Customer
4,P005,C,1,300,NaN,NaN,Board,B


A missing inspection record does not automatically mean zero defects. Missingness and zero are different concepts and must be interpreted using the business process.

## 27. Calculate Defect Rate

In [43]:
qa_data["Defect_Rate"] = (
    qa_data["Defect"] / qa_data["Produced"] * 100
)

qa_data

,Part_ID,Line,Shift,Produced,Measurement,Defect,Product,Customer,Defect_Rate
0,P001,A,1,500,10.01,0.0,Sensor,A,0.000000
1,P002,A,1,600,10.08,1.0,Sensor,B,0.166667
2,P003,B,2,450,9.99,0.0,Module,A,0.000000
3,P004,B,2,700,10.03,0.0,Module,C,0.000000
4,P005,C,1,300,NaN,NaN,Board,B,NaN


If inspection is missing, the defect-related fields may also be missing.

Do not blindly replace missing values with zero without understanding what the missingness means.

## 28. Merge + GroupBy

In [44]:
qa_data.groupby("Line")["Produced"].sum()

Line
A    1100
B    1150
C     300
Name: Produced, dtype: int64

In [45]:
qa_data.groupby("Line")["Defect"].sum()

Line
A    1.0
B    0.0
C    0.0
Name: Defect, dtype: float64

In [46]:
qa_data.groupby("Line")["Measurement"].mean()

Line
A    10.045
B    10.010
C       NaN
Name: Measurement, dtype: float64

This connects today's lesson with earlier lessons:

```text
Multiple Data Sources
        ↓
Merge
        ↓
Data Quality Checks
        ↓
Integrated Dataset
        ↓
GroupBy
        ↓
Pivot Table
        ↓
Visualization
        ↓
Business Insight
```

# 29. Professional Merge Workflow

```text
1. Understand the business relationship
          ↓
2. Identify the key
          ↓
3. Check key data types
          ↓
4. Check missing keys
          ↓
5. Check duplicate keys
          ↓
6. Choose join type
          ↓
7. Merge
          ↓
8. Validate row count
          ↓
9. Check unmatched records
          ↓
10. Validate business totals
```

A technically successful merge can still be analytically wrong.

# 30. Interview Questions

### Q1. Difference between `merge()` and `concat()`?
`merge()` matches records using keys; `concat()` stacks DataFrames by rows or columns.

### Q2. What is an inner join?
Keeps matching keys only.

### Q3. What is a left join?
Keeps all records from the left DataFrame and matching records from the right.

### Q4. What is one-to-many?
One record on one side can correspond to multiple records on the other side.

### Q5. What does `indicator=True` do?
Adds `_merge` showing `left_only`, `right_only`, or `both`.

### Q6. What does `validate="many_to_one"` do?
Checks that many left records match at most one right record.

### Q7. Why can a merge unexpectedly increase row count?
Duplicate keys can create one-to-many or many-to-many matches.

### Q8. What is the danger of a many-to-many merge?
Row multiplication can corrupt sums, counts, averages, and KPIs.

### Q9. Why check duplicate keys before merging?
To understand the relationship and prevent unintended row multiplication.

### Q10. What is `join()` commonly used for?
Convenient index-based joining.

# 31. Lesson 14 Assignment

## Part A — Basic Merge

Using `customers` and `orders`:

1. Perform an inner join.
2. Perform a left join.
3. Perform a right join.
4. Perform an outer join.
5. Use `indicator=True` and identify unmatched records.

## Part B — Manufacturing

Using `production`, `inspection`, and `part_master`:

6. Create an integrated QA dataset.
7. Use `validate="one_to_one"` where appropriate.
8. Find production records without inspection data.
9. Calculate defect rate.
10. Calculate total production by Line.
11. Calculate total defects by Line.
12. Calculate average measurement by Line.

# 32. Final Challenge

Create a final QA summary with:

```text
Line
Produced
Defects
Defect Rate
Average Measurement
```

Workflow:

```text
Production
     +
Inspection
     +
Part Master
     ↓
Integrated QA Dataset
     ↓
GroupBy Line
     ↓
Final QA Summary
```

Try to build it yourself before checking any solution.

In [47]:
# Practice area — solve the Final Challenge yourself.
#
# summary = ...
# summary["Defect_Rate"] = ...
# summary

# 33. Key Takeaways

```text
concat()
    ↓
Stack DataFrames

merge()
    ↓
Match DataFrames using keys

join()
    ↓
Convenient index-based joining
```

Join types:

```text
inner → matching only
left  → everything from left
right → everything from right
outer → everything from both
```

Most important principle:

> **A technically successful merge can still be analytically wrong.**

Always check:
- key quality
- duplicate keys
- expected relationship
- row count
- unmatched records
- business totals